# Viz Doc Chunker & Retriever

Interactive Notebook for testing Viz Document parsing, chunking, and storing and retrieving from XLake.

## Load and Parse One Document

In [7]:
from pathlib import Path

from xlake.utils.text_splitter import VizBibleDocSplitter

# Data Viz Bible base root
DVB_ROOT = Path("../../knowledge/data-viz-bible")

# Load a single document for initial testing
doc_path = DVB_ROOT / "00-index.md"
doc_text = doc_path.read_text()

print(f"Document: {doc_path.name}")
print(f"Total characters: {len(doc_text):,}")
print(f"Total lines: {len(doc_text.splitlines())}")
print("-" * 50)

# Create splitter and parse using split_file (handles origin automatically)
splitter = VizBibleDocSplitter()
chunks = splitter.split_file(doc_path)

Document: 00-index.md
Total characters: 8,467
Total lines: 129
--------------------------------------------------


## Parsing Stats

In [8]:
# Chunk statistics using new VizBibleChunk fields
print(f"Total chunks: {len(chunks)}")
print()

# Character count stats (using new char_count field)
char_counts = [c.char_count for c in chunks]
print("Character counts per chunk:")
print(f"  Min:  {min(char_counts):,}")
print(f"  Max:  {max(char_counts):,}")
print(f"  Avg:  {sum(char_counts) / len(char_counts):,.0f}")
print(f"  Total: {sum(char_counts):,}")
print()

# Section paths summary (using new section_path field)
print("Chunks by section path:")
for i, chunk in enumerate(chunks):
    stage = f"[{chunk.pipeline_stage}]" if chunk.pipeline_stage else "[ref]"
    print(f"  [{i}] {stage} {chunk.section_path} ({chunk.char_count:,} chars)")

Total chunks: 12

Character counts per chunk:
  Min:  125
  Max:  1,974
  Avg:  691
  Total: 8,289

Chunks by section path:
  [0] [ref] Data Viz Bible (125 chars)
  [1] [ref] Data Viz Bible > The Visualization Pipeline (603 chars)
  [2] [ref] Data Viz Bible > Quick Start (361 chars)
  [3] [ref] Data Viz Bible > Foundation Documents (401 chars)
  [4] [ref] Data Viz Bible > Action Interfaces (612 chars)
  [5] [ref] Data Viz Bible > Chart Types > P0 — Core (Always Available) (part 1) (1,974 chars)
  [6] [ref] Data Viz Bible > Chart Types > P0 — Core (Always Available) (part 2) (267 chars)
  [7] [ref] Data Viz Bible > Chart Types > P1 — Extended (1,829 chars)
  [8] [ref] Data Viz Bible > Chart Types > P2 — Specialized (917 chars)
  [9] [ref] Data Viz Bible > Worked Examples (540 chars)
  [10] [ref] Data Viz Bible > How to Use This Knowledge Base > For Single-Agent Architecture (264 chars)
  [11] [ref] Data Viz Bible > How to Use This Knowledge Base > For Multi-Agent Architecture (396 chars

## First Chunk, Second, Last Chunk

In [9]:
# Import helpers from lib
from lib.chunk_utils import print_chunk_details

print_chunk_details(chunks[0], "First Chunk")
print_chunk_details(chunks[1], "Second Chunk")
print_chunk_details(chunks[-1], "Last Chunk")

**First Chunk:**
  origin_path:    /data-viz-bible/00-index.md
  section_path:   Data Viz Bible
  document_type:  reference
  pipeline_stage: None
  chart_type:     None
  library:        None
  tags:           ['index', 'navigation', 'overview', 'pipeline']
  char_count:     125
  chunk_index:    0/12
------------------------------------------------------------
[Data Viz Bible]
# Data Viz Bible  
A comprehensive knowledge base for AI agents to select, refine, format, and implement data visualizations.


**Second Chunk:**
  origin_path:    /data-viz-bible/00-index.md
  section_path:   Data Viz Bible > The Visualization Pipeline
  document_type:  reference
  pipeline_stage: None
  chart_type:     None
  library:        None
  tags:           ['index', 'navigation', 'overview', 'pipeline']
  char_count:     603
  chunk_index:    1/12
------------------------------------------------------------
[The Visualization Pipeline]
## The Visualization Pipeline  
```
DataSchema + UserIntent
│
▼
┌─

## Load All the Data Viz Bible Documents

In [10]:
# Documents to load:
# - Core/Foundation (3): index, schema-reference, classification-system
# - Pipeline Interfaces/Blueprints (4): selection, refinement, formatting, implementation
# - All chart types across all pipeline stages

# Chart types with full pipeline coverage (selection + refinement + formatting + implementation)
CHART_TYPES_FULL = [
    "area-chart",
    "bar-chart-grouped",
    "bar-chart-horizontal",
    "bar-chart-stacked",
    "bar-chart-vertical",
    "boxplot",
    "data-table",
    "heatmap",
    "histogram",
    "kpi-card",
    "line-chart",
    "pie-donut-chart",
    "scatter-plot",
    "slope-chart",
    "treemap",
    "waterfall-chart",
]

# Chart types with selection only (no refinement/formatting/implementation docs yet)
CHART_TYPES_SELECTION_ONLY = [
    "bubble-chart",
    "bullet-chart",
    "funnel-chart",
    "lollipop-chart",
]

# Implementation library mappings (some charts use d3, others recharts/react)
IMPL_LIBRARY = {
    "area-chart": "recharts",
    "bar-chart-grouped": "recharts",
    "bar-chart-horizontal": "recharts",
    "bar-chart-stacked": "recharts",
    "bar-chart-vertical": "recharts",
    "boxplot": "d3",
    "bullet-chart": "d3",
    "data-table": "react",
    "funnel-chart": "d3",
    "heatmap": "d3",
    "histogram": "d3",
    "kpi-card": "react",
    "line-chart": "recharts",
    "lollipop-chart": "d3",
    "pie-donut-chart": "recharts",
    "scatter-plot": "recharts",
    "slope-chart": "d3",
    "treemap": "d3",
    "waterfall-chart": "d3",
}

DOCS_TO_LOAD = [
    # Core/Foundation documents (except 99-contributing.md)
    "00-index.md",
    "01-schema-reference.md",
    "02-classification-system.md",
    # Pipeline interface documents (blueprints)
    "selection/selection.md",
    "refinement/refinement.md",
    "formatting/formatting.md",
    "implementation/implementation.md",
]

# Add full pipeline documents for each chart type
for chart_type in CHART_TYPES_FULL:
    DOCS_TO_LOAD.extend([
        f"selection/selection-{chart_type}.md",
        f"refinement/refinement-{chart_type}.md",
        f"formatting/formatting-{chart_type}.md",
        f"implementation/implementation-{chart_type}-{IMPL_LIBRARY[chart_type]}.md",
    ])

# Add selection-only documents
for chart_type in CHART_TYPES_SELECTION_ONLY:
    DOCS_TO_LOAD.append(f"selection/selection-{chart_type}.md")
    # Add implementation if it exists
    if chart_type in IMPL_LIBRARY:
        DOCS_TO_LOAD.append(
            f"implementation/implementation-{chart_type}-{IMPL_LIBRARY[chart_type]}.md"
        )

print(f"Total documents to load: {len(DOCS_TO_LOAD)}")

# Load and parse all documents
all_chunks = []
doc_stats = []

for doc_rel_path in DOCS_TO_LOAD:
    doc_path = DVB_ROOT / doc_rel_path
    try:
        doc_chunks = splitter.split_file(doc_path)
        all_chunks.extend(doc_chunks)
        doc_stats.append({
            "path": doc_rel_path,
            "chunks": len(doc_chunks),
            "chars": sum(c.char_count for c in doc_chunks),
            "type": doc_chunks[0].document_type if doc_chunks else "unknown",
        })
        print(f"Loaded: {doc_rel_path} -> {len(doc_chunks)} chunks")
    except Exception as e:
        print(f"ERROR loading {doc_rel_path}: {e}")

print("-" * 60)
print(f"Total documents: {len(doc_stats)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Total characters: {sum(c.char_count for c in all_chunks):,}")

Total documents to load: 78
Loaded: 00-index.md -> 12 chunks
Loaded: 01-schema-reference.md -> 17 chunks
Loaded: 02-classification-system.md -> 15 chunks
Loaded: selection/selection.md -> 1 chunks
Loaded: refinement/refinement.md -> 1 chunks
Loaded: formatting/formatting.md -> 1 chunks
Loaded: implementation/implementation.md -> 1 chunks
Loaded: selection/selection-area-chart.md -> 14 chunks
Loaded: refinement/refinement-area-chart.md -> 10 chunks
Loaded: formatting/formatting-area-chart.md -> 8 chunks
Loaded: implementation/implementation-area-chart-recharts.md -> 8 chunks
Loaded: selection/selection-bar-chart-grouped.md -> 13 chunks
Loaded: refinement/refinement-bar-chart-grouped.md -> 11 chunks
Loaded: formatting/formatting-bar-chart-grouped.md -> 8 chunks
Loaded: implementation/implementation-bar-chart-grouped-recharts.md -> 7 chunks
Loaded: selection/selection-bar-chart-horizontal.md -> 17 chunks
Loaded: refinement/refinement-bar-chart-horizontal.md -> 23 chunks
Loaded: formatting

In [11]:
# Document stats summary
import pandas as pd

df_docs = pd.DataFrame(doc_stats)
print("Document Summary:")
print(df_docs.to_string(index=False))
print()

# Chunk breakdown by document type and pipeline stage
print("\nChunks by document_type:")
for doc_type in set(c.document_type for c in all_chunks):
    count = sum(1 for c in all_chunks if c.document_type == doc_type)
    print(f"  {doc_type}: {count}")

print("\nChunks by pipeline_stage:")
for stage in [None, "selection", "refinement", "formatting", "implementation"]:
    count = sum(1 for c in all_chunks if c.pipeline_stage == stage)
    label = stage if stage else "(none/reference)"
    print(f"  {label}: {count}")

Document Summary:
                                                          path  chunks  chars                  type
                                                   00-index.md      12   8289             reference
                                        01-schema-reference.md      17   7122             reference
                                   02-classification-system.md      15  12027             reference
                                        selection/selection.md       1   9568      action-interface
                                      refinement/refinement.md       1  19194      action-interface
                                      formatting/formatting.md       1  12217      action-interface
                              implementation/implementation.md       1  16443      action-interface
                             selection/selection-area-chart.md      14   2902 action-implementation
                           refinement/refinement-area-chart.md      10   2871 acti

## Set Up In-Memory CoreContextStore

In [12]:
# Import store setup helpers from lib
from lib.store_utils import create_test_contexts, create_test_store

# Create in-memory store for testing (no persistent files)
store = create_test_store()

# Admin context required for writes to CoreContextStore
tenant, user = create_test_contexts()

print("CoreContextStore initialized (in-memory mode)")
print(f"  Qdrant collection: {store.COLLECTION_VIZ_RULES}")
print(f"  Tenant: {tenant.identity.tenant_id}")
print(f"  User: {user.user_id} (role={user.role})")

CoreContextStore initialized (in-memory mode)
  Qdrant collection: core__viz_design_rules
  Tenant: actbi
  User: admin (role=admin)


## Index All Chunks

In [13]:
# Convert chunks to VizDesignRule and upsert to store
# This will embed each chunk and store it in Qdrant + SQLite

import time

print(f"Indexing {len(all_chunks)} chunks...")
start_time = time.time()

rule_ids = []
for i, chunk in enumerate(all_chunks):
    # Convert VizBibleChunk to VizDesignRule
    rule = chunk.to_viz_design_rule()
    # Upsert to store (creates embedding and stores in Qdrant + SQLite)
    rule_id = store.upsert_viz_design_rule(rule, tenant=tenant, user=user)
    rule_ids.append(rule_id)

    if (i + 1) % 10 == 0:
        print(f"  Indexed {i + 1}/{len(all_chunks)} chunks...")

elapsed = time.time() - start_time
print("-" * 60)
print(
    f"Indexed {len(rule_ids)} chunks in {elapsed:.2f}s ({len(rule_ids) / elapsed:.1f} chunks/sec)"
)

# Verify count in Qdrant
try:
    count = store._qdrant_client.count(store.COLLECTION_VIZ_RULES)
    print(f"Qdrant collection count: {count.count}")
except Exception as e:
    print(f"Could not get Qdrant count: {e}")

Indexing 987 chunks...
  Indexed 10/987 chunks...
  Indexed 20/987 chunks...
  Indexed 30/987 chunks...
  Indexed 40/987 chunks...
  Indexed 50/987 chunks...
  Indexed 60/987 chunks...
  Indexed 70/987 chunks...
  Indexed 80/987 chunks...
  Indexed 90/987 chunks...
  Indexed 100/987 chunks...
  Indexed 110/987 chunks...
  Indexed 120/987 chunks...
  Indexed 130/987 chunks...
  Indexed 140/987 chunks...
  Indexed 150/987 chunks...
  Indexed 160/987 chunks...
  Indexed 170/987 chunks...
  Indexed 180/987 chunks...
  Indexed 190/987 chunks...
  Indexed 200/987 chunks...
  Indexed 210/987 chunks...
  Indexed 220/987 chunks...
  Indexed 230/987 chunks...
  Indexed 240/987 chunks...
  Indexed 250/987 chunks...
  Indexed 260/987 chunks...
  Indexed 270/987 chunks...
  Indexed 280/987 chunks...
  Indexed 290/987 chunks...
  Indexed 300/987 chunks...
  Indexed 310/987 chunks...
  Indexed 320/987 chunks...
  Indexed 330/987 chunks...
  Indexed 340/987 chunks...
  Indexed 350/987 chunks...
  Inde

## Search with Debug Metadata

In [14]:
# Import search helpers from lib
from lib.retrieval_utils import print_search_results, search_with_scores


# Create a wrapper that uses the current store/tenant/user
def search(query: str, top_k: int = 5) -> list[dict]:
    """Search wrapper using current store and contexts."""
    return search_with_scores(store, query, tenant, user, top_k)


print("search() wrapper and print_search_results() imported from lib.")

search() wrapper and print_search_results() imported from lib.


## Test Queries

In [15]:
# Test Query 1: When to use horizontal bar charts
query1 = "when should I use a horizontal bar chart?"
results1 = search(query1, top_k=5)
print_search_results(query1, results1)

Query: "when should I use a horizontal bar chart?"
Results: 5
[1] SCORE=0.8804
    origin:   /data-viz-bible/selection/selection-bar-chart-horizontal.md
    section:  Selection: Bar Chart (Horizontal) > When to Use
    stage:    selection | chart: bar-chart-horizontal | lib: (none)
    preview:  [Bar Chart Horizontal - Selection - When to Use] ## When to Use   - **Ranking data**: Showing items ...

[2] SCORE=0.8624
    origin:   /data-viz-bible/selection/selection-bar-chart-horizontal.md
    section:  Selection: Bar Chart (Horizontal) > When NOT to Use
    stage:    selection | chart: bar-chart-horizontal | lib: (none)
    preview:  [Bar Chart Horizontal - Selection - When NOT to Use] ## When NOT to Use   | Scenario | Problem | Use...

[3] SCORE=0.8409
    origin:   /data-viz-bible/selection/selection-bar-chart-horizontal.md
    section:  Selection: Bar Chart (Horizontal) > When to Use > Data Pattern
    stage:    selection | chart: bar-chart-horizontal | lib: (none)
    preview:  [Bar

In [16]:
# Test Query 2: Formatting axis labels
query2 = "how to format axis labels for long category names?"
results2 = search(query2, top_k=5)
print_search_results(query2, results2)

Query: "how to format axis labels for long category names?"
Results: 5
[1] SCORE=0.7942
    origin:   /data-viz-bible/formatting/formatting-bar-chart-horizontal.md
    section:  Formatting: Bar Chart (Horizontal) > Axis Styling > Y-Axis (Categories)
    stage:    formatting | chart: bar-chart-horizontal | lib: (none)
    preview:  [Bar Chart Horizontal - Formatting - Y-Axis (Categories)] ### Y-Axis (Categories)   | Element | Styl...

[2] SCORE=0.7880
    origin:   /data-viz-bible/formatting/formatting-histogram.md
    section:  Formatting: Histogram > Axis Styling
    stage:    formatting | chart: histogram | lib: (none)
    preview:  [Histogram - Formatting - Axis Styling] ## Axis Styling   | Element | Style | |---------|-------| | ...

[3] SCORE=0.7804
    origin:   /data-viz-bible/formatting/formatting-heatmap.md
    section:  Formatting: Heatmap > Axis Labels
    stage:    formatting | chart: heatmap | lib: (none)
    preview:  [Heatmap - Formatting - Axis Labels] ## Axis Labels   

In [17]:
# Test Query 3: Recharts implementation
query3 = "implementation options for horizontal bars. Do NOT use recharts"
results3 = search(query3, top_k=5)
print_search_results(query3, results3)

Query: "implementation options for horizontal bars. Do NOT use recharts"
Results: 5
[1] SCORE=0.8388
    origin:   /data-viz-bible/implementation/implementation-bar-chart-horizontal-recharts.md
    section:  Implementation: Bar Chart Horizontal (Recharts) > Dependencies
    stage:    implementation | chart: bar-chart-horizontal | lib: recharts
    preview:  [Bar Chart Horizontal - Implementation - recharts - Dependencies] ## Dependencies   ```javascript im...

[2] SCORE=0.8371
    origin:   /data-viz-bible/implementation/implementation-bar-chart-horizontal-recharts.md
    section:  Implementation: Bar Chart Horizontal (Recharts) > Props Reference > BarChart Props (Horizontal)
    stage:    implementation | chart: bar-chart-horizontal | lib: recharts
    preview:  [Bar Chart Horizontal - Implementation - recharts - BarChart Props (Horizontal)] ## Props Reference ...

[3] SCORE=0.8269
    origin:   /data-viz-bible/implementation/implementation-bar-chart-horizontal-recharts.md
    section

## Evaluation: Test Data Chart Type Retrieval

Test retrieval accuracy using `nlp_chart_test_data.yaml`. For each test case:
1. Build query from `nlp_query` + `output_schema` fields
2. Search for relevant chunks
3. Compare retrieved chart types with expected chart type

In [18]:
# Load test cases from YAML
import yaml
from typing import Any

with open("../data_samples/nlp_chart_test_data.yaml") as f:
    test_data = yaml.safe_load(f)

test_cases: list[dict[str, Any]] = test_data["test_cases"]
print(f"Loaded {len(test_cases)} test cases:")
for tc in test_cases:
    expected = tc["expected_chart"]["chart_type"]
    print(f"  - {tc['id']}: expected chart_type = '{expected}'")

Loaded 5 test cases:
  - tc_001_monthly_sales: expected chart_type = 'bar-chart-vertical'
  - tc_002_sales_by_region_product: expected chart_type = 'heatmap'
  - tc_003_customer_cohort_retention: expected chart_type = 'cohort-heatmap'
  - tc_004_revenue_vs_cost_trend: expected chart_type = 'combo'
  - tc_005_distribution_analysis: expected chart_type = 'boxplot'


In [19]:
# Import evaluation helpers
from lib.retrieval_utils import (
    build_search_query,
    content_mentions_data_shape,
    get_combo_chart_types,
    get_data_shape,
    matches_expected_chart_type,
)

# Define instruction variants to test
INSTRUCTION_VARIANTS = {
    "no_instructions": None,
    "match_data_shape": "Match data pattern for this query",
}

# Collect results per variant
all_eval_results = {}

for variant_name, instructions in INSTRUCTION_VARIANTS.items():
    print(f"\n{'=' * 60}")
    print(f"Running variant: {variant_name}")
    print(f"{'=' * 60}")

    eval_results = []

    for tc in test_cases:
        query = build_search_query(tc, instructions=instructions)
        results = search(query, top_k=5)

        expected_type = tc["expected_chart"]["chart_type"]
        data_shape = get_data_shape(tc)

        # Chart type matching
        retrieved_types = [
            r["chart_type"]
            for r in results
            if r.get("chart_type") and r["chart_type"] != "(none)"
        ]

        # For combo charts, compare list of unique retrieved types against combo_types
        # For single charts, check if any of top 3 matches expected
        if expected_type == "combo":
            combo_types = get_combo_chart_types(tc)
            unique_retrieved = list(set(retrieved_types))
            chart_match = matches_expected_chart_type(unique_retrieved, combo_types)
        else:
            top_3_types = retrieved_types[:3]
            chart_match = any(
                matches_expected_chart_type(ct, expected_type) for ct in top_3_types
            )

        # Data shape matching
        shape_matches = sum(
            1
            for r in results[:5]
            if content_mentions_data_shape(r.get("content_preview", ""), data_shape)
        )

        # Extract chunk metadata for each retrieved result
        retrieved_chunks = [
            {
                "rank": r["rank"],
                "score": r["score"],
                "origin": r["origin_path"],
                "section": r["section_path"],
                "chart_type": r["chart_type"],
                "stage": r["pipeline_stage"],
            }
            for r in results[:5]
        ]

        eval_results.append({
            "id": tc["id"],
            "expected_chart": expected_type,
            "data_shape": data_shape,
            "chart_type_match": chart_match,
            "data_shape_match": shape_matches > 0,
            "data_shape_count": shape_matches,
            "top_score": results[0]["score"] if results else 0.0,
            "retrieved_chunks": retrieved_chunks,
        })

        # Print per-case result with chunk details
        chart_status = "PASS" if chart_match else "FAIL"
        shape_status = "PASS" if shape_matches > 0 else "FAIL"
        print(f"\n[chart:{chart_status}|shape:{shape_status}] {tc['id']}")
        print(f"  Expected: chart={expected_type}, shape={data_shape}")
        print("  Retrieved chunks (top 5):")
        for chunk in retrieved_chunks:
            print(
                f"    [{chunk['rank']}] score={chunk['score']:.4f} | {chunk['chart_type']} | {chunk['stage']}"
            )
            print(f"        origin: {chunk['origin']}")
            print(f"        section: {chunk['section']}")

    all_eval_results[variant_name] = eval_results

print(f"\n{'=' * 60}")
print("Batched evaluation complete.")


Running variant: no_instructions

[chart:FAIL|shape:PASS] tc_001_monthly_sales
  Expected: chart=bar-chart-vertical, shape=1D + 1M
  Retrieved chunks (top 5):
    [1] score=0.7380 | (none) | (none)
        origin: /data-viz-bible/01-schema-reference.md
        section: Schema Reference > Example Schema
    [2] score=0.7354 | (none) | (none)
        origin: /data-viz-bible/01-schema-reference.md
        section: Schema Reference > Data Pattern Syntax
    [3] score=0.7170 | (none) | (none)
        origin: /data-viz-bible/01-schema-reference.md
        section: Schema Reference > Schema to Chart Mapping > Table Mapping
    [4] score=0.6967 | line-chart | selection
        origin: /data-viz-bible/selection/selection-line-chart.md
        section: Selection: Line Chart > When to Use > Data Pattern
    [5] score=0.6923 | area-chart | selection
        origin: /data-viz-bible/selection/selection-area-chart.md
        section: Selection: Area Chart > When to Use > Data Pattern

[chart:FAIL|sh

In [20]:
# Build results matrix per matching category
import pandas as pd

# Build matrix for chart_type_match
chart_match_data = {
    tc["id"]: {
        variant: next(r["chart_type_match"] for r in results if r["id"] == tc["id"])
        for variant, results in all_eval_results.items()
    }
    for tc in test_cases
}
df_chart_match = pd.DataFrame(chart_match_data).T
df_chart_match.columns.name = "variant"
df_chart_match.index.name = "test_case"

# Build matrix for data_shape_match
shape_match_data = {
    tc["id"]: {
        variant: next(r["data_shape_match"] for r in results if r["id"] == tc["id"])
        for variant, results in all_eval_results.items()
    }
    for tc in test_cases
}
df_shape_match = pd.DataFrame(shape_match_data).T
df_shape_match.columns.name = "variant"
df_shape_match.index.name = "test_case"

# Display results
print("=== Chart Type Match Matrix ===")
print(df_chart_match.replace({True: "PASS", False: "FAIL"}))
print(f"\nTotals: {df_chart_match.sum().to_dict()}")

print("\n=== Data Shape Match Matrix ===")
print(df_shape_match.replace({True: "PASS", False: "FAIL"}))
print(f"\nTotals: {df_shape_match.sum().to_dict()}")

=== Chart Type Match Matrix ===
variant                          no_instructions match_data_shape
test_case                                                        
tc_001_monthly_sales                        FAIL             FAIL
tc_002_sales_by_region_product              FAIL             FAIL
tc_003_customer_cohort_retention            FAIL             FAIL
tc_004_revenue_vs_cost_trend                FAIL             FAIL
tc_005_distribution_analysis                PASS             FAIL

Totals: {'no_instructions': 1, 'match_data_shape': 0}

=== Data Shape Match Matrix ===
variant                          no_instructions match_data_shape
test_case                                                        
tc_001_monthly_sales                        PASS             PASS
tc_002_sales_by_region_product              FAIL             FAIL
tc_003_customer_cohort_retention            FAIL             FAIL
tc_004_revenue_vs_cost_trend                FAIL             FAIL
tc_005_distribution_an

### Summary by Variant

In [21]:
# Combined summary
summary_rows = []
for variant in INSTRUCTION_VARIANTS:
    results = all_eval_results[variant]
    summary_rows.append({
        "variant": variant,
        "chart_type_pass": sum(r["chart_type_match"] for r in results),
        "data_shape_pass": sum(r["data_shape_match"] for r in results),
        "total_cases": len(results),
    })

df_summary = pd.DataFrame(summary_rows)
print("=== Summary by Variant ===")
print(df_summary.to_string(index=False))

=== Summary by Variant ===
         variant  chart_type_pass  data_shape_pass  total_cases
 no_instructions                1                1            5
match_data_shape                0                2            5


### Notes on Evaluation Results

- **Expected failures**: Test cases for `heatmap`, `cohort_heatmap`, `combo`, and `box` charts will fail because we only loaded horizontal bar chart documents. The knowledge base doesn't contain documents for these chart types yet.
- **Expected pass**: Test case `tc_001_monthly_sales` (bar chart) should pass since `bar` maps to `bar-chart-horizontal` (among other bar variants).
- **Data shape matching**: Checks if retrieved content mentions the data pattern (e.g., "1D + 1M" for 1 DIMENSION + 1 MEASURE).
- To improve coverage, load documents for additional chart types.